In [1]:
# Import libraries
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder

In [2]:
# Load dataset
df = pd.read_csv('/home/ae7ba225-76ef-4005-aee2-7847a70630ed/SilentTelecomChurn/CustomerChurn.csv')
df.head()

,LoyaltyID,Customer ID,Senior Citizen,Partner,Dependents,Tenure,Phone Service,Multiple Lines,Internet Service,Online Security,...,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn
0,318537,7590-VHVEG,No,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,152148,5575-GNVDE,No,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,326527,3668-QPYBK,No,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,845894,7795-CFOCW,No,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,503388,9237-HQITU,No,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Fix Total Charges and drop missing values
df["TotalCharges"] = pd.to_numeric(df["Total Charges"], errors="coerce")
df = df.dropna()
df.drop(columns=["TotalCharges"], inplace=True)
df = df.drop("Customer ID", axis=1)

In [4]:
# Encode target variable
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

In [5]:
# Select the 7 columns — 7 predictors + target
df_final = df[[
    "Tenure",
    "Monthly Charges",
    "Multiple Lines",
    "Tech Support",
    "Internet Service",
    "Online Security",
    "Contract",
    "Churn"
]].copy()

df_final.head()

,Tenure,Monthly Charges,Multiple Lines,Tech Support,Internet Service,Online Security,Contract,Churn
0,1,29.85,No phone service,No,DSL,No,Month-to-month,0
1,34,56.95,No,No,DSL,Yes,One year,0
2,2,53.85,No,No,DSL,Yes,Month-to-month,1
3,45,42.30,No phone service,Yes,DSL,Yes,One year,0
4,2,70.70,No,No,Fiber optic,No,Month-to-month,1


In [6]:
# 1. TENURE GROUP
# <= 12 months = New, <= 36 months = Established, > 36 months = Loyal
def tenure_group(x):
    if x <= 12:
        return "New"
    elif x <= 36:
        return "Established"
    else:
        return "Loyal"

df_final["Tenure Group"] = df_final["Tenure"].apply(tenure_group)

print(df_final["Tenure Group"].value_counts())

Tenure Group
Loyal          3001
New            2175
Established    1856
Name: count, dtype: int64


In [7]:
# 2. CHARGE LEVEL
# Monthly Charges < 35 = Low, < 70 = Medium, >= 70 = High
def charge_level(x):
    if x < 35:
        return "Low"
    elif x < 70:
        return "Medium"
    else:
        return "High"

df_final["Charge Level"] = df_final["Monthly Charges"].apply(charge_level)

print(df_final["Charge Level"].value_counts())

Charge Level
High      3589
Low       1725
Medium    1718
Name: count, dtype: int64


In [8]:
# 3. CLEAN AND ENCODE MULTIPLE LINES
# 'No phone service' and 'No' both become 0, 'Yes' becomes 1
df_final["Multiple Lines"] = df_final["Multiple Lines"].replace("No phone service", "No")

df_final["Multiple Lines"] = df_final["Multiple Lines"].map({
    "No": 0,
    "Yes": 1
})

print(df_final["Multiple Lines"].value_counts(dropna=False))
print("NaN count:", df_final["Multiple Lines"].isnull().sum(), "<-- should be 0")

Multiple Lines
0    4065
1    2967
Name: count, dtype: int64
NaN count: 0 <-- should be 0


In [9]:
# 4. TECH SUPPORT
# No internet service and No both become 0, Yes becomes 1
df_final["Tech Support"] = df_final["Tech Support"].replace("No internet service", "No")
df_final["Tech Support"] = df_final["Tech Support"].map({"No": 0, "Yes": 1})
print(df_final["Tech Support"].value_counts(dropna=False))

Tech Support
0.0    4992
1.0    2039
NaN       1
Name: count, dtype: int64


In [10]:
# 5. INTERNET SERVICE
# No = 0, DSL = 1, Fiber optic = 2
df_final["Internet Service"] = df_final["Internet Service"].map({
    "No": 0,
    "DSL": 1,
    "Fiber optic": 2
})
print(df_final["Internet Service"].value_counts(dropna=False))

Internet Service
2    3096
1    2416
0    1520
Name: count, dtype: int64


In [11]:
# 6. ONLINE SECURITY
# No internet service and No both become 0, Yes becomes 1
df_final["Online Security"] = df_final["Online Security"].replace("No internet service", "No")
df_final["Online Security"] = df_final["Online Security"].map({"No": 0, "Yes": 1})
print(df_final["Online Security"].value_counts(dropna=False))

Online Security
0    5017
1    2015
Name: count, dtype: int64


In [12]:
# 7. CONTRACT
# Month-to-month = 0, One year = 1, Two year = 2
df_final["Contract"] = df_final["Contract"].map({
    "Month-to-month": 0,
    "One year": 1,
    "Two year": 2
})
print(df_final["Contract"].value_counts(dropna=False))

Contract
0    3875
2    1685
1    1472
Name: count, dtype: int64


In [13]:
# 4. ENCODE TENURE GROUP AND CHARGE LEVEL
# Using manual map so numbers match our defined order (not alphabetical)
# New=0, Established=1, Loyal=2
# Low=0, Medium=1, High=2

df_final["Tenure Group"] = df_final["Tenure Group"].map({
    "New": 0,
    "Established": 1,
    "Loyal": 2
})

df_final["Charge Level"] = df_final["Charge Level"].map({
    "Low": 0,
    "Medium": 1,
    "High": 2
})

# Churn is already 0/1 from earlier — no encoding needed
print("Tenure Group unique values:", sorted(df_final["Tenure Group"].unique()))
print("Charge Level unique values:", sorted(df_final["Charge Level"].unique()))

Tenure Group unique values: [np.int64(0), np.int64(1), np.int64(2)]
Charge Level unique values: [np.int64(0), np.int64(1), np.int64(2)]


In [21]:
# Fix missing value in Tech Support — fill with 0 (No) since it is just 1 row
df_final["Tech Support"] = df_final["Tech Support"].fillna(0)

print("Missing values after fix:")
print(df_final["Tech Support"].isnull().sum(), "<-- should be 0")

Missing values after fix:
0 <-- should be 0


In [22]:
# Fix decimal encoding — convert all encoded columns to integer
df_final["Tech Support"] = df_final["Tech Support"].astype(int)
df_final["Multiple Lines"] = df_final["Multiple Lines"].astype(int)
df_final["Internet Service"] = df_final["Internet Service"].astype(int)
df_final["Online Security"] = df_final["Online Security"].astype(int)
df_final["Contract"] = df_final["Contract"].astype(int)
df_final["Tenure Group"] = df_final["Tenure Group"].astype(int)
df_final["Charge Level"] = df_final["Charge Level"].astype(int)

print("Data types after fix:")
print(df_final.dtypes)

Data types after fix:
Tenure                int64
Monthly Charges     float64
Multiple Lines        int64
Tech Support          int64
Internet Service      int64
Online Security       int64
Contract              int64
Churn                 int64
Tenure Group          int64
Charge Level          int64
dtype: object


In [23]:
# FINAL DATASET
df_model = df_final[[
    "Tenure Group",
    "Charge Level",
    "Multiple Lines",
    "Tech Support",
    "Internet Service",
    "Online Security",
    "Contract",
    "Churn"
]].copy()

print("Shape:", df_model.shape)
print("Missing values:")
print(df_model.isnull().sum())

Shape: (7032, 8)
Missing values:
Tenure Group        0
Charge Level        0
Multiple Lines      0
Tech Support        0
Internet Service    0
Online Security     0
Contract            0
Churn               0
dtype: int64


In [24]:
df_model.head(10)

,Tenure Group,Charge Level,Multiple Lines,Tech Support,Internet Service,Online Security,Contract,Churn
0,0,0,0,0,1,0,0,0
1,1,1,0,0,1,1,1,0
2,0,1,0,0,1,1,0,1
3,2,1,0,1,1,1,1,0
4,0,2,0,0,2,0,0,1
5,0,2,1,0,2,0,0,1
6,1,2,1,0,2,0,0,0
7,0,0,0,0,1,1,0,0
8,1,2,1,1,2,0,0,1
9,2,1,0,0,1,1,1,0


In [25]:
# Save
df_model.to_csv("/home/ae7ba225-76ef-4005-aee2-7847a70630ed/SilentTelecomChurn/engineered_features.csv", index=False)
print("Saved: updatedengineered_features.csv")

Saved: updatedengineered_features.csv
